In [51]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import json

def load_glove(path):
    glove_dict = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc="Loading GloVe"):
            values = line.split()
            word = values[0]
            vector = np.array(values[1:], dtype='float32')
            glove_dict[word] = vector
    return glove_dict


In [52]:
def sentence_to_vec(sentence, glove, dim=100):
    words = sentence.split()
    vectors = [glove[w] for w in words if w in glove]

    if len(vectors) == 0:
        return np.zeros(dim)

    return np.mean(vectors, axis=0)

In [53]:
def process_schema(file_path, glove, text_column="text", dim=100):
    df = pd.read_csv(file_path)

    embeddings = []
    
    for text in tqdm(df[text_column].astype(str), desc=f"Processing {file_path}"):
        vec = sentence_to_vec(text, glove, dim)
        embeddings.append(vec)

    df["embedding"] = embeddings
    return df

In [54]:
def save_outputs(df, output_prefix):
    # Save as CSV (vectors flattened)
    df_csv = df.copy()
    df_csv["embedding"] = df_csv["embedding"].apply(lambda x: ",".join(map(str, x)))
    df_csv.to_csv(f"{output_prefix}.csv", index=False)

    # Save as JSON
    df_json = df.copy()
    df_json["embedding"] = df_json["embedding"].apply(lambda x: x.tolist())
    df_json.to_json(f"{output_prefix}.json", orient="records", indent=2)

In [55]:
if __name__ == "__main__":

    GLOVE_PATH = r"C:\Users\MAZEN M. NASR\Desktop\glove.6B.100d.txt"

    schema_files = {
    "schema1": r"C:\Users\MAZEN M. NASR\Desktop\SCHEMA_1.csv",
    "schema2": r"C:\Users\MAZEN M. NASR\Desktop\SCHEMA_2.csv",
    "schema3": r"C:\Users\MAZEN M. NASR\Desktop\SCHEMA_3.csv"
    }

    TEXT_COLUMN = "review_text"   
    DIM = 100

    df = pd.read_csv(r"C:\Users\MAZEN M. NASR\Desktop\SCHEMA_1.csv")
    print(df.columns)

    print("Loading GloVe...")
    glove = load_glove(GLOVE_PATH)


    for name, path in schema_files.items():
        print(f"\nProcessing {name}...")

        df = process_schema(path, glove, TEXT_COLUMN, DIM)

        save_outputs(df, fr"C:\Users\MAZEN M. NASR\Desktop\glove_{name}")

    print("\n✅ All schemas processed successfully!")

Index(['Unnamed: 0', 'game_name', 'app_id', 'review_text', 'review_length',
       'hours_played', 'review_date', 'owners', 'developers', 'publishers',
       'genres', 'platforms', 'categories', 'release_date', 'price'],
      dtype='object')
Loading GloVe...


Loading GloVe: 400000it [00:08, 48287.87it/s]



Processing schema1...


Processing C:\Users\MAZEN M. NASR\Desktop\SCHEMA_1.csv: 100%|█████████████████████| 198/198 [00:00<00:00, 13095.42it/s]



Processing schema2...


Processing C:\Users\MAZEN M. NASR\Desktop\SCHEMA_2.csv: 100%|█████████████████████| 197/197 [00:00<00:00, 15893.32it/s]



Processing schema3...


Processing C:\Users\MAZEN M. NASR\Desktop\SCHEMA_3.csv: 100%|█████████████████████| 194/194 [00:00<00:00, 21462.73it/s]


✅ All schemas processed successfully!
